# 📸 Урок 16 — Твой проект «Классификатор моих фото»

Сегодня ты обучишь классификатор СВОИХ фото и сделаешь сайт, ссылку на который можно отправить родителям.

> ★ **Оценивается (10 баллов):** свои данные (2) · аугментация (2) · transfer learning (3) · Gradio + ссылка (3).

## Шаг 1 · Выбери категории и сними фото ✍️
Выбери 2–3 понятные категории и сними по 15–20 фото каждой (разный фон и ракурс!).

*Мои категории:* …

Загрузи фото в папки (одна папка = один класс):
```
data/
  класс1/  ...
  класс2/  ...
```

## Шаг 2 · Загрузка данных + аугментация
Аугментация из одного фото делает много вариантов — модель учится лучше.

In [ ]:
import tensorflow as tf
from tensorflow import keras
train = keras.utils.image_dataset_from_directory('data', image_size=(160,160), batch_size=16)
class_names = train.class_names
print('Категории:', class_names)
augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),   # отражение
    keras.layers.RandomRotation(0.1),          # поворот
    keras.layers.RandomZoom(0.1),              # приближение
])

## Шаг 3 · Обучаем на готовой сети (transfer learning)
Берём MobileNet (уже видел миллионы картинок) и доучиваем под свои классы.

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(160,160,3), include_top=False, weights='imagenet')
base.trainable = False
model = keras.Sequential([
    augment,
    keras.layers.Rescaling(1./127.5, offset=-1),
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(len(class_names), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train, epochs=8)

## Шаг 4 · Сделай сайт через Gradio 🌐
Получишь публичную ссылку — открой её и загрузи новое фото!

In [ ]:
!pip install gradio -q
import gradio as gr, tensorflow as tf
def predict(img):
    x = tf.image.resize(img, (160,160))[None, ...]
    p = model.predict(x)[0]
    return {class_names[i]: float(p[i]) for i in range(len(class_names))}
gr.Interface(fn=predict, inputs=gr.Image(), outputs=gr.Label(num_top_classes=3),
             title='Классификатор моих фото').launch(share=True)   # share=True -> публичная ссылка

✍️ **Ответь.** Где твоя модель путается? Помогла ли аугментация?

*Ответ:* …

## ✅ Проверь себя
1. Зачем аугментация?
2. Почему transfer learning работает даже на 20 фото?
3. Что делает `share=True` в Gradio?

<details><summary>Ответы</summary>

1. Из немногих фото делает много вариантов — больше данных для обучения.
2. Готовая сеть уже умеет узнавать узоры; мы доучиваем только 'голову'.
3. Даёт публичную ссылку на твой веб-интерфейс.
</details>

---
### 🏁 Чек-лист перед сдачей
- [ ] 2–3 категории, по 15–20 фото
- [ ] Применена аугментация
- [ ] Модель обучена (transfer learning) и работает на новых фото
- [ ] Работает Gradio-интерфейс с публичной ссылкой
- [ ] Записано, где модель ошибается

🎉 Отправь ссылку родителям — пусть проверят твой классификатор!